# DeepFaceLive — GPU (Google Colab) 🚀

Run your face-swap app on a **free Colab GPU** so the celebrity deepfake is **smooth**, not laggy.

### How to use (about 3 minutes)
1. Top menu: **Runtime → Change runtime type → T4 GPU → Save**.
2. Run each cell below **in order** — click the ▶ button on the left of each cell and wait for it to finish.
3. When the **last cell** prints a link ending in **`.trycloudflare.com`**, open **that link on your phone**.
4. Pick a celebrity model → **Load Model** → **Start Camera**.

> Notes: the first model you load downloads a file (~100 MB), so give it a few seconds. Keep this Colab tab open while you use the app — when you close it, the link stops working (that is normal, just re-run the cells next time).

In [ ]:
#@title 1) Check the GPU is switched on
# If this shows no GPU: Runtime -> Change runtime type -> T4 GPU -> Save, then re-run.
!nvidia-smi -L || echo "No GPU found — set Runtime -> Change runtime type -> T4 GPU, then re-run this cell."

In [ ]:
#@title 2) Download the app & install the GPU runtime (about 2-3 min)
%cd /content
!rm -rf Deepfaketrial
!git clone --depth 1 -b colab-gpu https://github.com/oluwacoded/Deepfaketrial.git
%cd /content/Deepfaketrial

# GPU build of onnxruntime + the exact CUDA/cuDNN libraries it needs, plus the web deps.
# Installing the nvidia-*-cu12 wheels avoids the common "libcudnn.so.9 not found" error.
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1
!pip -q install onnxruntime-gpu nvidia-cuda-runtime-cu12 nvidia-cudnn-cu12 nvidia-cublas-cu12 nvidia-cufft-cu12 nvidia-curand-cu12 nvidia-cusparse-cu12 nvidia-cuda-nvrtc-cu12 nvidia-nvjitlink-cu12 flask flask-socketio eventlet numexpr h5py onnx

# onnxruntime can only find those libraries if they are on the library path.
# Build that path here so the app (next cell) can actually use the GPU.
import os, glob, site
_lib_dirs = []
for _sp in set(site.getsitepackages() + [site.getusersitepackages()]):
    _lib_dirs += glob.glob(os.path.join(_sp, 'nvidia', '*', 'lib'))
GPU_LIB_PATH = ':'.join(_lib_dirs)
os.environ['LD_LIBRARY_PATH'] = GPU_LIB_PATH + ':' + os.environ.get('LD_LIBRARY_PATH', '')

# cloudflared gives us a public https link the phone can open.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Check the GPU provider in a FRESH process (so it picks up the library path above).
import subprocess, sys, json
try:
    _out = subprocess.run(
        [sys.executable, '-c', 'import onnxruntime as o, json; print(json.dumps(o.get_available_providers()))'],
        env=dict(os.environ), capture_output=True, text=True, timeout=180)
    _lines = _out.stdout.strip().splitlines()
    _last = _lines[-1] if _lines else ''
    _provs = json.loads(_last) if _last.startswith('[') else []
    print()
    if 'CUDAExecutionProvider' in _provs:
        print('GPU is ready for inference. ✅')
    else:
        print('GPU provider not active yet — the app will still run on CPU (slower). ⚠️')
        print('   Make sure Runtime -> Change runtime type -> T4 GPU is set, then re-run this cell.')
        _err = _out.stderr.strip().splitlines()
        if _err:
            print('   Last message:', _err[-1])
except Exception as _e:
    print()
    print('Could not verify onnxruntime (the app will still try):', _e)

In [ ]:
#@title 3) Start the app — then open the printed link on your PHONE
import subprocess, threading, re, time, sys, os, glob, site

os.chdir('/content/Deepfaketrial')

# Put the GPU libraries on the path for the server process too.
_lib_dirs = []
for _sp in set(site.getsitepackages() + [site.getusersitepackages()]):
    _lib_dirs += glob.glob(os.path.join(_sp, 'nvidia', '*', 'lib'))
env = dict(os.environ)
env['LD_LIBRARY_PATH'] = ':'.join(_lib_dirs) + ':' + env.get('LD_LIBRARY_PATH', '')

def _drain(proc, tag):
    for line in proc.stdout:
        print(tag, line, end='')

# 1) start the face-swap server on port 5000 (uses the GPU automatically)
server = subprocess.Popen([sys.executable, 'web_server.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
threading.Thread(target=_drain, args=(server, '[server]'), daemon=True).start()

print('Starting the face-swap server (loading the model)...')
time.sleep(15)

if server.poll() is not None:
    print('=' * 64)
    print('The server stopped before it could start.')
    print('Read the [server] lines above — the last one says why.')
    print('=' * 64)
else:
    # 2) open a public https tunnel to it
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    public_url = None
    deadline = time.time() + 40
    while time.time() < deadline:
        line = tunnel.stdout.readline()
        if not line:
            break
        m = re.search(r'https://[-a-z0-9]+[.]trycloudflare[.]com', line)
        if m:
            public_url = m.group(0)
            break
    threading.Thread(target=_drain, args=(tunnel, '[tunnel]'), daemon=True).start()
    print('=' * 64)
    if public_url:
        print('  OPEN THIS LINK ON YOUR PHONE:')
        print('      ' + public_url)
    else:
        print('  Could not read the link automatically.')
        print('  Look in the [tunnel] lines above for a .trycloudflare.com link, or re-run this cell.')
    print('=' * 64)
    print('Keep this Colab tab open while you use the app.')